# download-aic — stream-extract zip video giao thông, không lưu zip

Mỗi shard xử lý các file có chỉ số trong `[tu, den)` (zip ~10-11 GB, đĩa kernel
~21 GB nên KHÔNG tải zip về: stream từng member qua `stream-unzip` thẳng vào
`OUT/<tên>/`. Đỉnh đĩa chỉ còn ~1× dung lượng video thay vì ~2×.
Ghi một dòng manifest mỗi file xong, rồi ghi mốc `.moc.json` cho CLI ghép.
Lỗi mạng bao thành `ConnectionError` (nằm trong MANG_RE của CLI) để tự đổi acc;lỗi khác (vd hết đĩa) để traceback nguyên cho dễ đọc.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 1, "commit": "", "noi_tiep": None, "output": "manifest.jsonl", "base": "https://aic-data.ledo.io.vn", "files": ["Video_N001-N010"]}

In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stream-unzip"], check=True)
from stream_unzip import stream_unzip
import urllib.request
from urllib.error import URLError, HTTPError

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
ra = OUT / SHARD["output"]

if SHARD.get("noi_tiep"):
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print("noi tiep manifest:", f)

def done_ids():
    ids = set()
    if ra.exists():
        for line in ra.open(encoding="utf-8"):
            try:
                ids.add(json.loads(line)["id"])
            except Exception:
                pass
    return ids

UA = {"User-Agent": "Mozilla/5.0"}

def content_length(url):
    req = urllib.request.Request(url, headers=UA, method="HEAD")
    with urllib.request.urlopen(req, timeout=30) as r:
        return int(r.headers.get("Content-Length") or 0)

def fetch(url, chunk=1 << 20):
    try:
        r = urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30)
    except (URLError, HTTPError, OSError) as e:
        raise ConnectionError(f"mo mang {url}: {e}")
    with r:
        while True:
            b = r.read(chunk)
            if not b:
                break
            yield b

def extract(url, d):
    d.mkdir(parents=True, exist_ok=True)
    n_f, n_b = 0, 0
    for raw, size, chunks in stream_unzip(fetch(url)):
        name = raw.decode("utf-8") if isinstance(raw, bytes) else raw
        parts = [x for x in name.replace(chr(92), "/").split("/") if x not in ("", ".", "..")]
        if not parts:
            for _ in chunks:
                pass
            continue
        dst = d.joinpath(*parts)
        if dst.is_file() and dst.stat().st_size == size:
            for _ in chunks:
                pass
            n_f += 1; n_b += size
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        with dst.open("wb") as f:
            for c in chunks:
                f.write(c)
        n_f += 1; n_b += dst.stat().st_size
    assert n_f, f"{d.name} bung ra rong"
    return n_f, n_b

xong = done_ids()
with ra.open("a", encoding="utf-8") as f:
    for k in range(SHARD["tu"], SHARD["den"]):
        idk = f"n{k:03d}"
        name = SHARD["files"][k]
        d = OUT / name
        files = [x for x in d.rglob("*") if x.is_file()] if d.is_dir() else []
        if idk in xong and files:
            continue
        url = f"{SHARD['base']}/{name}.zip"
        # mp4 gan nhu khong nen: can free > kich thuoc zip. Fail som thay vi giua chung.
        cl = content_length(url)
        free = shutil.disk_usage(OUT).free
        assert cl == 0 or free > cl, f"het dia: free {free} < zip {cl}"
        err = None
        for lan in range(3):
            try:
                n_f, n_b = extract(url, d)
                break
            except (ConnectionError, HTTPError, URLError) as e:
                err = e
                print(f"mang loi lan {lan + 1}: {e}")
        else:
            raise ConnectionError(f"{name} hong 3 lan: {err}")
        f.write(json.dumps({"id": idk, "dir": name, "files": n_f, "bytes": n_b}) + "\n")
        f.flush()
        print("xong", name, n_f, n_b)

# ponytail: moc inline thay vì import ghi_moc từ src — job không cần dataset src nào.
moc = {"tu": SHARD["tu"], "den": SHARD["den"], "so_dong": len(done_ids()), "xong": True}
Path(str(ra) + ".moc.json").write_text(json.dumps(moc))
print(moc)
